In [13]:
import sys
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

sys.path.insert(0, os.getcwd())
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import joblib
from src.data_utils import load_config, load_raw_data, clean_data, save_processed
from src.features import encode_categoricals, scale_features, engineer_features, get_feature_matrix

config = load_config("config/config.yaml")
df_raw = load_raw_data(config)

Dataset cargado: 20718 filas, 27 columnas


In [14]:

df_clean = clean_data(df_raw, config)
print(f"\nShape final: {df_clean.shape}")

Duplicados eliminados: 82
Canciones virales (>= 150,000,000): 4687
viral
0    0.766386
1    0.233614
Name: proportion, dtype: float64

Shape final: (20063, 28)


In [15]:

df_feat = engineer_features(df_clean)
print("Features creadas:")
print([c for c in df_feat.columns if c not in df_raw.columns])

Features creadas:
['viral', 'engagement_rate', 'log_Views', 'log_Likes', 'log_Comments', 'log_Stream', 'popularity_score']


In [16]:
df_feat

,Artist,Url_spotify,Track,Album,Album_type,Uri,Danceability,Energy,Key,Loudness,...,Licensed,official_video,Stream,viral,engagement_rate,log_Views,log_Likes,log_Comments,log_Stream,popularity_score
0,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Feel Good Inc.,Demon Days,album,spotify:track:0d28khcov6AiegSCpG5TuT,0.818,0.705,6.0,-6.679,...,True,True,1.040235e+09,1,0.009215,20.357341,15.643425,12.043012,20.762712,20.600564
1,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Rhinestone Eyes,Plastic Beach,album,spotify:track:1foMv2HQwfQ2vntFf9HFeG,0.676,0.703,8.0,-5.815,...,True,True,3.100837e+08,1,0.015416,18.092338,13.891665,10.341872,19.552353,18.968347
2,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,New Gold (feat. Tame Impala and Bootie Brown),New Gold (feat. Tame Impala and Bootie Brown),single,spotify:track:64dLd6rVqDLtkXFYrEUHIU,0.695,0.923,1.0,-3.930,...,True,True,6.306347e+07,0,0.034326,15.947907,12.550169,8.909235,17.959652,17.154954
3,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,On Melancholy Hill,Plastic Beach,album,spotify:track:0q6LuUqGLUiCPP1cbdwFs3,0.689,0.739,2.0,-5.810,...,True,True,4.346636e+08,1,0.008707,19.170940,14.396931,10.919262,19.890083,19.602426
4,Gorillaz,https://open.spotify.com/artist/3AA28KZvwAUcZu...,Clint Eastwood,Gorillaz,album,spotify:track:7yMiX7n9SBvadzox8T5jzT,0.663,0.694,10.0,-8.627,...,True,True,6.172597e+08,1,0.010272,20.242777,15.639627,11.957169,20.240800,20.241591
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20713,SICK LEGEND,https://open.spotify.com/artist/3EYY5FwDkHEYLw...,JUST DANCE HARDSTYLE,JUST DANCE HARDSTYLE,single,spotify:track:0RtcKQGyI4hr8FgFH1TuYG,0.582,0.926,5.0,-6.344,...,True,True,9.227144e+06,0,0.015528,11.179953,7.015712,0.000000,16.037660,14.094577
20714,SICK LEGEND,https://open.spotify.com/artist/3EYY5FwDkHEYLw...,SET FIRE TO THE RAIN HARDSTYLE,SET FIRE TO THE RAIN HARDSTYLE,single,spotify:track:3rHvPA8lUnPBkaLyPOc0VV,0.531,0.936,4.0,-1.786,...,True,True,1.089818e+07,0,0.012256,12.012136,7.610853,0.000000,16.204106,14.527318
20715,SICK LEGEND,https://open.spotify.com/artist/3EYY5FwDkHEYLw...,OUTSIDE HARDSTYLE SPED UP,OUTSIDE HARDSTYLE SPED UP,single,spotify:track:4jk00YxPtPbhvHJE9N4ddv,0.443,0.830,4.0,-4.679,...,True,True,6.226110e+06,0,0.009229,10.481420,5.799093,0.000000,15.644262,13.579126
20716,SICK LEGEND,https://open.spotify.com/artist/3EYY5FwDkHEYLw...,ONLY GIRL HARDSTYLE,ONLY GIRL HARDSTYLE,single,spotify:track:5EyErbpsugWliX006eTDex,0.417,0.767,9.0,-4.004,...,True,True,6.873961e+06,0,0.013468,8.784775,4.488636,0.000000,15.743251,12.959861


In [17]:
df_feat, encoders = encode_categoricals(df_feat, config["features_categoricas"])
joblib.dump(encoders, "models/label_encoders.pkl")
print(" Encoders guardados")

 Encoders guardados


In [18]:
y_clf = df_feat[config["target_classification"]]

In [19]:
X = get_feature_matrix(df_feat, config, use_log=True)

print("Shape X:", X.shape)

Shape X: (20063, 14)


In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train_c, y_test_c = train_test_split(
    X, y_clf,
    test_size=config["test_size"],
    random_state=config["random_state"],
    stratify=y_clf
)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Balance viral en train: {y_train_c.mean():.2%}")

Train: (16050, 14) | Test: (4013, 14)
Balance viral en train: 23.36%


In [21]:
from sklearn.impute import SimpleImputer
import pandas as pd

imputer = SimpleImputer(strategy="mean")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

In [22]:
X_train_sc, X_test_sc, scaler = scale_features(X_train, X_test)

In [23]:
import os
import joblib
from pathlib import Path

project_root = Path.cwd()
if project_root.name == 'notebooks':
    project_root = project_root.parent

save_path = project_root / 'data' / 'processed' / 'splits.pkl'
save_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(
    (X_train, X_test, X_train_sc, X_test_sc,
     y_train_c, y_test_c,
     list(X.columns)),
    save_path
)

print("Splits guardados correctamente")

Splits guardados correctamente


 Datos guardados en data/processed/datos_limpios.csv
viral
0    0.766386
1    0.233614
Name: proportion, dtype: float64

Preparación completada. Datos listos para modelado.
